# 5.2 一次看多少例子？

# 第 5 章：訓練與評估形成一個流程

前置：第4章的 logits／shift／loss。程式預設只做結構、梯度與記錄檢查。凡是涉及學到能力的對照，先給可執行配方，之後由讀者在 GPU 上跑；不會把隨機模型的輸出寫成訓練成果。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：一次更新的生命週期**

忘記清零會累加；累積梯度時才刻意保留。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/backward.svg")))

**先想一想：**一筆有10個答案token，另一筆只有1個，可以各佔一半嗎？

mini-batch同時計算數筆例子，再把有效token的loss平均。較大的batch常讓梯度波動變小，但同一步用更多資料與記憶體。

**把直覺寫成數學：**平均loss的梯度等於各loss梯度的平均；不同長度例子應依有效token數加權。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.model import loss_sum

logits = torch.randn(2, 4, 5, requires_grad=True)
labels = torch.tensor([[1, 2, 3, 4], [1, -100, -100, -100]])
total, count = loss_sum(logits, labels)
print("有效token", count.item(), "平均loss", (total / count).item())

**如何讀結果：**總共有5個有效token，不是batch大小2。第16.6章用同樣分母累積梯度。

**只改一件事：**只加一個ignored padding位置。
